## Status

> **Work in progress** — Price data is a representative manual sample (3–5 products 
> per brand). Full range audit pending — see options below.

**Current limitation:** Representative sample only. Conclusions on price positioning 
and the demographic gap are valid; range depth analysis is not.

**Future enhancement (TODO):**
- Option A: Manual full-range audit — go through each brand's catalogue and record 
  every product and price into a CSV, then replace manual_data with pd.read_csv()
- Option B: Selenium scraping — renders JavaScript before parsing, enabling automated 
  collection of full M&S/Next/Boden ranges

# 01 — Competitor Pricing Analysis

## Objective
Map the competitive pricing landscape for loungewear and pyjama brands across UK and US markets. Identify price point gaps, particularly in the 11–19 age demographic.

## Brands Analysed

### UK Brands
| Brand | Positioning | Primary Demographic |
|---|---|---|
| SHEIN | Ultra-budget | Kids (8–12) |
| George (Asda) | Budget | Kids (1–14) |
| Primark | Budget | Kids (5–15) |
| H&M Kids | Mass market | Kids (8–14) |
| Matalan | Mass market | Kids (4–13) |
| M&S | Mass market | Kids & teens (1–16) |
| Next | Mass market | Kids & teens (3–16) |
| Zara Kids | Mid-market | Kids (6–14) |
| River Island Kids | Mid-market | Kids (5–14) |
| Asos (own brand) | Mid-market | Adult (20–55) |
| Brandy Melville | Lifestyle / teen | Young adult (14–30) |
| Lounge | Lifestyle / DTC | Adult (20–55) |
| Boden | Premium casual | Kids (2–12) + adult |
| White Fox | Streetwear-adjacent | Young adult (16–30) |
| Desmond & Dempsey | Luxury | Adult (25–45) + kids (0–10) |

### US Brands (benchmarking)
| Brand | Positioning | Primary Demographic |
|---|---|---|
| Pink Chicken | Premium | Kids (0–12) + adult |
| Katie J NYC | Premium | Kids/teen (4–16) |
| Roller Rabbit | Luxury | Adult + kids (0–12) |
| Eberjey | Luxury | Adult + kids |

## Method
M&S: live scraping via BeautifulSoup (server-side rendered). All other brands: manual collection from public product pages, February–March 2026. USD converted to GBP at 0.79.

---

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import re
import time
import os
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid')
COLOURS = {'uk': '#2E5FA3', 'us': '#E8A838', 'gap': '#C0392B', 'neutral': '#7F8C8D'}
USD_GBP = 0.79

# Resolve notebook directory (works in VS Code and standard Jupyter)
try:
    _nb_dir = os.path.dirname(__vsc_ipynb_file__)
except NameError:
    _nb_dir = '/Users/sewabreiler/Documents/kidswear-market-analysis/01_competitor_analysis'

print(f'Ready. Analysis date: {datetime.today().strftime("%d %B %Y")}')
print(f'USD/GBP rate: {USD_GBP}')
print(f'Notebook dir: {_nb_dir}')

---
## Section 1: M&S — Live Scraping

M&S girls nightwear (ages 1–16) is server-side rendered, making it accessible via BeautifulSoup without JavaScript execution. Product names appear in `<h2>` tags; prices appear as text nodes in the surrounding product card.

In [2]:
def scrape_ms_nightwear(url, delay=2):
    """
    Scrape product names and prices from M&S girls nightwear listing page.
    Returns a DataFrame with columns: brand, market, product, price_gbp, min_age, max_age, source.
    """
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                             'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    time.sleep(delay)
    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Request failed: {e}')
        return pd.DataFrame()

    soup = BeautifulSoup(r.text, 'html.parser')
    products = []

    for h2 in soup.find_all('h2'):
        name = h2.get_text(strip=True)
        if not name or len(name) < 5:
            continue

        # Search for price in surrounding siblings and parent
        price = None
        for sibling in h2.find_previous_siblings():
            text = sibling.get_text(strip=True)
            if re.search(r'£\d', text):
                matches = re.findall(r'£(\d+(?:\.\d{2})?)', text)
                if matches:
                    price = float(matches[0])  # take lower bound if range
                    break
        if not price and h2.parent:
            parent_text = h2.parent.get_text(separator=' ', strip=True)
            matches = re.findall(r'£(\d+(?:\.\d{2})?)', parent_text)
            if matches:
                price = float(matches[0])

        # Extract age range from product name e.g. '(6-16 Yrs)'
        age = re.search(r'\((\d+)-(\d+)\s*(?:Yrs|Months)?\)', name, re.IGNORECASE)
        min_age = int(age.group(1)) if age else None
        max_age = int(age.group(2)) if age else None

        products.append({'brand': 'M&S', 'market': 'UK', 'demographic': 'Kids/Teen',
                         'product': name, 'price_gbp': price,
                         'min_age': min_age, 'max_age': max_age, 'source': 'scraped'})

    df = pd.DataFrame(products).dropna(subset=['price_gbp'])
    print(f'Scraped {len(df)} products from M&S')
    return df


df_ms_scraped = scrape_ms_nightwear('https://www.marksandspencer.com/l/kids/girls/nightwear')

if not df_ms_scraped.empty:
    print(df_ms_scraped[['product', 'price_gbp', 'min_age', 'max_age']].head(8).to_string())
else:
    print('No scraped data — manual M&S prices used in dataset below')

Scraped 0 products from M&S
No scraped data — manual M&S prices used in dataset below


In [3]:
# NOTE: M&S product listings are rendered client-side via JavaScript.
# BeautifulSoup retrieves the page shell but product cards are not present
# in the raw HTML response. Manual price data is used instead.
# Future enhancement: implement Selenium/Playwright for JavaScript rendering.

---
## Section 2: Full Price Dataset — All Brands

In [ ]:
# Load from the updated CSV (source of truth — edit the CSV to add/change brands)
CSV_PATH = os.path.join(_nb_dir, 'data', 'competitor_prices_all.csv')
print(f'Loading from: {CSV_PATH}')
assert os.path.exists(CSV_PATH), f'❌ File not found: {CSV_PATH}'

df_all = pd.read_csv(CSV_PATH)

# If M&S was also scraped successfully, merge (scraping usually returns 0 products)
if not df_ms_scraped.empty:
    df_ms_scraped['demographic'] = 'Kids/Teen'
    df_all = pd.concat([df_all[df_all['brand'] != 'M&S'], df_ms_scraped], ignore_index=True)
    print(f'Using scraped M&S data ({len(df_ms_scraped)} products)')

print(f'Dataset: {len(df_all)} products, {df_all["brand"].nunique()} brands')
df_all.groupby(['brand', 'market'])['price_gbp'].agg(['min', 'max', 'mean', 'count']).round(2)

---
## Section 3: UK vs US Price Ranges

In [ ]:
brand_summary = df_all.groupby(['brand', 'market'])['price_gbp'].agg(
    min_price='min', max_price='max', mean_price='mean'
).round(2).reset_index().sort_values(['market', 'mean_price'], ascending=[True, False])

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=False)

for ax, market in zip(axes, ['UK', 'US']):
    data = brand_summary[brand_summary['market'] == market].reset_index(drop=True)
    colour = COLOURS['uk'] if market == 'UK' else COLOURS['us']

    for i, row in data.iterrows():
        ax.barh(i, row['max_price'] - row['min_price'],
                left=row['min_price'], height=0.5, color=colour, alpha=0.6)
        ax.scatter(row['mean_price'], i, color='#2C3E50', zorder=5, s=70)
        ax.text(row['max_price'] + 2, i, f"£{row['mean_price']:.0f} avg",
                va='center', fontsize=9, color='#555')

    ax.axvspan(55, 85, alpha=0.08, color=COLOURS['gap'], label='UK target entry zone (£55–£85)')
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data['brand'].tolist(), fontsize=11)
    ax.set_xlabel('Price (£ GBP)', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('£%d'))
    title = f'{market} Brands' if market == 'UK' else 'US Brands (USD→GBP @ 0.79)'
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Competitor Price Ranges — UK vs US\n(bar = range, dot = mean)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(_nb_dir, 'chart_01_price_ranges_uk_us.png'), dpi=150, bbox_inches='tight')
plt.show()

### Chart Commentary — UK vs US Price Ranges

**UK market** is split into two clear tiers with almost nothing in between. The mass-market cluster (SHEIN, George, Primark, H&M, Matalan, M&S, Next) sits between £6–28. Boden sits just above at £32–36. Then there is a large gap before Desmond & Dempsey (£89 for kids), which is positioned as a luxury gift product rather than a brand teenagers would buy for themselves.

**US market** tells a different story: Katie J NYC (£62–75) and Pink Chicken (£54–87) occupy a genuine mid-premium space aimed at kids and teens — a segment that has no UK equivalent. Roller Rabbit and Eberjey operate at luxury adult price points with kid lines attached.

**Key implication:** The UK market has a structural pricing gap between ~£35 and £89 in kids/teen nightwear. US brands validate that demand exists at £55–75 for this demographic. A UK brand entering at £20–40 would sit at the top of the accessible range — above mass market on quality and identity, without requiring a luxury spend.

In [11]:
# Current file directory
import os
print(os.getcwd())

/Users/sewabreiler/anaconda_projects/473af41e-a720-4ec6-8df3-8c61fd7da1fe


---
## Section 4: Age Coverage Map

### Chart Commentary — Age Coverage Map

The age coverage map reveals a consistent pattern across UK brands: **coverage effectively ends at age 14**. M&S and Next extend to 16, but their product offering at that age is functionally identical to what they sell to a 6-year-old — there is no distinct identity or aesthetic shift for the older teen.

**The 11–19 gap (shaded red)** is not simply a sizing gap — it is an identity gap. Brands serving this age group either:
- Are extensions of adult brands (White Fox, Lounge, Brandy Melville) that don't cater below 14–16
- Are mass-market kids brands (M&S, Next) that lose relevance as teens develop a stronger sense of style

**Desmond & Dempsey kids** only goes to age 10, making it a parent-purchase gift product, not something a teenager would choose themselves.

**US comparison:** Katie J NYC is the only brand in the dataset with a dedicated teen-facing product line extending to age 16 — and it has no UK equivalent. This reinforces the gap as a structural market opportunity rather than a deliberate choice by incumbents.

In [ ]:
coverage = {
    'M&S (UK)':                [(1, 16, 'UK', 'primary')],
    'Next (UK)':               [(3, 16, 'UK', 'primary')],
    'Boden (UK)':              [(2, 12, 'UK', 'primary'), (18, 65, 'UK', 'secondary')],
    'Desmond & Dempsey (UK)':  [(25, 55, 'UK', 'primary'), (2, 10, 'UK', 'secondary')],
    'White Fox (UK)':          [(16, 30, 'UK', 'primary')],
    'H&M Kids (UK)':           [(4, 14, 'UK', 'primary')],
    'Zara Kids (UK)':          [(4, 14, 'UK', 'primary')],
    'River Island Kids (UK)':  [(4, 14, 'UK', 'primary')],
    'Primark (UK)':            [(1, 15, 'UK', 'primary')],
    'George/Matalan (UK)':     [(1, 14, 'UK', 'primary')],
    'SHEIN (UK)':              [(4, 14, 'UK', 'primary')],
    'Lounge (UK)':             [(18, 45, 'UK', 'primary')],
    'Brandy Melville (UK)':    [(13, 25, 'UK', 'primary')],
    'Roller Rabbit (US)':      [(25, 55, 'US', 'primary'), (2, 12, 'US', 'secondary')],
    'Pink Chicken (US)':       [(2, 12, 'US', 'primary'), (25, 45, 'US', 'secondary')],
    'Katie J NYC (US)':        [(4, 16, 'US', 'primary')],
    'Eberjey (US)':            [(25, 55, 'US', 'primary'), (2, 12, 'US', 'secondary')],
}

colour_map = {
    ('UK', 'primary'): '#2E5FA3', ('UK', 'secondary'): '#A8C4E0',
    ('US', 'primary'): '#E8A838', ('US', 'secondary'): '#F5D99A',
}

brand_list = list(coverage.keys())
uk_count = sum(1 for b in brand_list if '(UK)' in b)

fig, ax = plt.subplots(figsize=(14, 9))

for i, brand in enumerate(brand_list):
    for (min_a, max_a, market, focus) in coverage[brand]:
        ax.barh(i, max_a - min_a, left=min_a, height=0.55,
                color=colour_map[(market, focus)], alpha=0.9)

# Gap shading
ax.axvspan(11, 19, alpha=0.12, color=COLOURS['gap'])
ax.axvline(11, color=COLOURS['gap'], lw=1.5, ls='--', alpha=0.7)
ax.axvline(19, color=COLOURS['gap'], lw=1.5, ls='--', alpha=0.7)
ax.text(15, -0.9, 'UK gap: 11–19', color=COLOURS['gap'], fontsize=10, fontweight='bold', ha='center')

# Divider UK / US
ax.axhline(uk_count - 0.5, color='grey', lw=0.8, ls=':')
ax.text(60, uk_count - 0.3, '← UK brands  /  US brands →', color='grey', fontsize=8)

for age in range(0, 65, 5):
    ax.axvline(age, color='grey', lw=0.3, alpha=0.4)

ax.set_yticks(range(len(brand_list)))
ax.set_yticklabels(brand_list, fontsize=10)
ax.set_xlabel('Age', fontsize=11)
ax.set_xlim(0, 65)
ax.set_title('Age Range Coverage by Brand — Primary (solid) vs Secondary (light)',
             fontsize=13, fontweight='bold', pad=15)

legend_elements = [
    mpatches.Patch(color='#2E5FA3', alpha=0.9, label='UK primary'),
    mpatches.Patch(color='#A8C4E0', alpha=0.9, label='UK secondary'),
    mpatches.Patch(color='#E8A838', alpha=0.9, label='US primary'),
    mpatches.Patch(color='#F5D99A', alpha=0.9, label='US secondary'),
    mpatches.Patch(color=COLOURS['gap'], alpha=0.2, label='UK gap: 11–19'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(_nb_dir, 'chart_02_age_coverage.png'), dpi=150, bbox_inches='tight')
plt.show()

### Chart Commentary — The UK Price Cliff

This chart makes the structural gap explicit. Every UK kids/teen brand in the dataset prices between **£6 and £36**. Above £36, there is nothing until Desmond & Dempsey Kids at £89 — a gap of over £50.

**Mass market cluster (£6–28):** SHEIN, George, Primark, H&M, Matalan, M&S, and Next all sit here. These brands compete primarily on price and availability. None has a distinct teen identity or aesthetic — they are functional, not aspirational.

**Boden (£32–36):** The ceiling of the accessible range. Strong visual identity and good quality but targeted at younger kids (2–12) and parent-driven purchasing. Does not address the teen consumer directly.

**The gap (£36–£89):** No UK brand occupies this space for kids or teens. A brand entering at £25–40 with genuine aesthetic differentiation — quality fabrics, considered design, social-native marketing — would face no direct UK competition in this segment.

**Desmond & Dempsey Kids (£89):** A luxury price point, stops at age 10, and purchased as a gift. Not a competitor to a teen-facing brand — it occupies a completely different purchase occasion.

**What this means for positioning:** A new entrant at £25–35 sits above the mass-market ceiling (£28) but well below the luxury threshold. The opportunity is to own the space between "good enough" and "unaffordable" — where brand identity, not price, is the reason to buy.

---
## Section 5: The UK Price Cliff

In [ ]:
uk_kids = df_all[
    (df_all['market'] == 'UK') &
    (df_all['demographic'].isin(['Kids', 'Kids/Teen', 'Teen']))
].copy()

# Summarise to one row per brand: min, max, mean price
brand_sum = (
    uk_kids.groupby('brand')['price_gbp']
    .agg(min_p='min', max_p='max', mean_p='mean')
    .reset_index()
    .sort_values('mean_p')
    .reset_index(drop=True)
)

# Assign segment colours
SEGMENT = {
    'SHEIN':              '#9E9E9E',
    'George (Asda)':      '#9E9E9E',
    'Primark':            '#9E9E9E',
    'H&M Kids':           '#78909C',
    'Matalan':            '#78909C',
    'M&S':                '#78909C',
    'Next':               '#78909C',
    'Zara Kids':          '#5C7A9F',
    'River Island Kids':  '#5C7A9F',
    'Boden':              '#2E5FA3',
    'Desmond & Dempsey':  '#C0392B',
}
default_colour = '#7F8C8D'

fig, ax = plt.subplots(figsize=(13, 7))

GAP_LO, GAP_HI = 36, 89   # the price cliff gap

# Shade the gap
ax.axvspan(GAP_LO, GAP_HI, alpha=0.10, color='#C0392B', zorder=0)
ax.axvline(GAP_LO, color='#C0392B', lw=1.2, ls='--', alpha=0.6, zorder=1)
ax.axvline(GAP_HI, color='#C0392B', lw=1.2, ls='--', alpha=0.6, zorder=1)

for i, row in brand_sum.iterrows():
    colour = SEGMENT.get(row['brand'], default_colour)
    bar_lo = row['min_p']
    bar_hi = row['max_p']
    # Range bar
    ax.barh(i, bar_hi - bar_lo, left=bar_lo, height=0.55,
            color=colour, alpha=0.75, zorder=2)
    # Mean dot
    ax.scatter(row['mean_p'], i, color='#1a1a1a', s=55, zorder=4)
    # Price label at right end
    ax.text(bar_hi + 1.5, i, f"£{row['min_p']:.0f}–£{bar_hi:.0f}",
            va='center', fontsize=8.5, color='#444')

# Gap annotation
mid_gap = (GAP_LO + GAP_HI) / 2
ax.text(mid_gap, len(brand_sum) - 0.2,
        f'No UK kids brand\n£{GAP_LO}–£{GAP_HI}',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
        color='#C0392B',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#C0392B', alpha=0.85))

ax.set_yticks(range(len(brand_sum)))
ax.set_yticklabels(brand_sum['brand'].tolist(), fontsize=10)
ax.set_xlabel('Price (£ GBP)', fontsize=11)
ax.set_xlim(0, 115)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('£%d'))
ax.set_title('UK Kids & Teen Nightwear — Price Ranges by Brand\n(bar = min–max range, dot = mean)',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig(os.path.join(_nb_dir, 'chart_03_price_cliff.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6: Key Findings

In [ ]:
print('=' * 65)
print('COMPETITOR PRICING ANALYSIS — KEY FINDINGS')
print('=' * 65)

uk = df_all[df_all['market'] == 'UK']
us = df_all[df_all['market'] == 'US']

print()
print('UK BRANDS — PRICE SUMMARY')
print(uk.groupby('brand')['price_gbp'].agg(['min','max','mean','count']).round(2).to_string())

print()
print('US BRANDS — PRICE SUMMARY')
print(us.groupby('brand')['price_gbp'].agg(['min','max','mean','count']).round(2).to_string())

print()
print(f'Total: {len(df_all)} products across {df_all["brand"].nunique()} brands')
print('=' * 65)